# 8장. MCP로 같은 업무 도구 제공하기

같은 목표 조회를 sandbox 내부의 stdio MCP로 제공합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | MCP 서버 연결·초기화·실제 호출 이력 |
| 이번 실습 조합 | CSV 분석 + stdio MCP 목표 조회 + 목표·달성률 응답 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

| 구분 | 7장 Function Tool | 이 장의 stdio MCP |
| --- | --- | --- |
| 목표 조회 실행 위치 | 노트북의 Python 애플리케이션 | sandbox의 별도 MCP 프로세스 |
| 도구 등록 | 함수 스키마와 로컬 핸들러 | 서버 실행·연결 설정 |
| 호출 처리 | SDK의 `tool_handlers` | Agents API의 MCP 연결 |
| 목표 파일 위치 | 로컬 JSON | sandbox에 배치한 JSON |

노트북은 서버 코드를 읽고 요청을 구성합니다. 실제 서버 프로세스는 세션 실행 환경의 executor가 시작합니다. 본문에서 서버를 직접 실행하거나 `tool_handlers`·수동 `tool_result` 처리를 추가하지 않습니다.

수동 실습은 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_extensions.py --chapter 8 --live`로 실행합니다. 서버 초기화·도구 호출·분석 완료·정답 검증을 구분하세요.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 환경 준비와 실행 모드

로컬에서는 `uv sync --locked` 후 `.venv` 커널을 선택합니다. Colab에서는 다운로드 안내의 독립 실행본을 업로드합니다. 준비 셀은 패키지를 직접 설치하고 포함 파일을 생성합니다.
아래 RUN_LIVE를 True로 바꿨을 때만 실제 API 호출·sandbox 생성이 발생합니다.
기존 `.env.local`의 키·모델을 재사용하며 키는 sandbox·ZIP·로그에 넣지 않습니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
import json
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
CHAPTER = "08_mcp"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print({"live_enabled": settings.enabled, "model": settings.model or "Offline mode"})


## 1. 서버 코드와 공급할 파일 확인하기

`examples/mcp/sales_targets_server.py`의 FastMCP 서버를 읽습니다. `@server.tool()`이 함수를 공개하고 `--data`로 데이터 경로를 받으며, 마지막에 stdio transport로 실행합니다.

아래 셀은 파일을 읽고 코드와 입력 CSV를 **표시만 합니다**. 서버를 노트북에서 직접 실행하지 않습니다. 직접 실행하면 stdio 입력을 기다리며 계속 실행 중일 수 있습니다. MCP 서버의 stdout은 프로토콜 통신용이므로 일반 디버그 출력을 섞지 않고 로그는 stderr로 보냅니다.

이 서버는 기간 형식·존재 여부를 검사합니다. 7장의 애플리케이션 역할 검사가 MCP로 자동 이전되는 것은 아닙니다.


In [ ]:
MCP_VERSION = "1.26.0"
input_path = settings.root / "data/sales.csv"
server_path = settings.root / "examples/mcp/sales_targets_server.py"
targets_path = settings.root / "data/sales_targets.json"
csv_bytes = input_path.read_bytes()
csv_text = csv_bytes.decode("utf-8")
server_bytes = server_path.read_bytes()
targets_bytes = targets_path.read_bytes()
print(csv_text)
print("MCP server source (not executed here):")
print(server_bytes.decode("utf-8"))


### 로컬 경로와 sandbox 경로 연결하기

| 로컬 파일 | sandbox 목적지 | 사용 위치 |
| --- | --- | --- |
| `data/sales.csv` | `/workspace/sales.csv` | 에이전트의 Python 분석 |
| `examples/mcp/sales_targets_server.py` | `/workspace/sales_targets_server.py` | MCP 실행 인자 |
| `data/sales_targets.json` | `/workspace/sales_targets.json` | 서버의 `--data` 인자 |

목표 JSON도 같은 sandbox에 놓습니다. 따라서 이 구성은 목표 데이터를 에이전트 코드와 격리하거나 도구 결과의 독립적인 출처를 증명하는 방식이 아닙니다. 합성 데이터로 연결 방법을 배우는 실습입니다. 일반적인 사내 자격 증명·파일 격리 설계는 별도로 검토해야 합니다.


## 2. 환경·MCP 연결·분석 요청 구성하기

먼저 세 파일과 원격 `mcp` 패키지를 환경에 공급합니다. 로컬/Colab의 패키지와 sandbox 패키지는 별개입니다. `MCP_VERSION` 한 곳에서 원격 요청 버전을 지정합니다.

현재 공식 가이드는 OpenAI-hosted stdio MCP에서 네트워크 정책을 생략하거나 `enabled`로 설정하도록 안내하며 `disabled`·`restricted`는 지원하지 않습니다. 이 예제는 `enabled`를 명시합니다. stdio 서버 자체는 외부 HTTP 포트를 열지 않지만 네트워크 정책을 끈 환경이라는 뜻은 아닙니다. [공식 MCP 가이드](https://developers.openai.com/api/docs/guides/agents-api/tools/mcp)

다음 셀은 요청을 구성할 뿐 패키지를 설치하거나 서버를 실행하지 않습니다.


In [ ]:
import base64

environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "enabled"},
    "packages": {"python": [f"mcp=={MCP_VERSION}"]},
    "files": [
        {"type": "inline", "path": "/workspace/sales.csv",
         "data": base64.b64encode(csv_bytes).decode("ascii")},
        {"type": "inline", "path": "/workspace/sales_targets_server.py",
         "data": base64.b64encode(server_bytes).decode("ascii")},
        {"type": "inline", "path": "/workspace/sales_targets.json",
         "data": base64.b64encode(targets_bytes).decode("ascii")},
    ],
}
print({"packages": environment["packages"], "network": environment["network"],
       "files": [{"path": file["path"], "bytes": len(base64.b64decode(file["data"]))}
                 for file in environment["files"]]})


### MCP 연결 필드 이해하기

| 필드 | 역할 |
| --- | --- |
| `server_label` | Agents API 호출 이력에서 서버를 식별할 이름 |
| `command`, `args` | sandbox에서 실행할 프로그램과 인자 |
| `cwd` | 서버 프로세스의 기존 절대 작업 디렉터리 |
| `allowed_tools` | 발견·호출하도록 허용할 도구 |
| `required=True` | 서버 초기화 실패 시 Turn 실패 처리 |

`required=True`는 초기화 요구이며 도구를 한 번 호출하라는 뜻이 아닙니다. 실제 호출 조건은 입력 메시지와 저장된 이력으로 따로 확인합니다. stdio에는 HTTP 연결 출발점을 지정하는 `connection_origin`을 넣지 않습니다.

`server_label`은 API에서 정하는 연결 이름이며 FastMCP 서버 자체의 이름과 구분합니다. 이 실습은 외부 MCP 자격 증명을 사용하지 않습니다. [공식 설정 설명](https://developers.openai.com/api/docs/guides/agents-api/tools/mcp)


In [ ]:
mcp_tool = {
    "type": "mcp",
    "server_label": "sales_targets",
    "transport": {
        "type": "stdio",
        "command": "python",
        "args": ["/workspace/sales_targets_server.py", "--data", "/workspace/sales_targets.json"],
        "cwd": "/workspace",
    },
    "allowed_tools": ["get_sales_target"],
    "required": True,
}
agent = {
    "model": settings.model,
    "instructions": (
        "You are a careful sales analyst. Use Python for all calculations. "
        "Use only the provided CSV; never invent rows. Do not browse or delegate. "
        "Keep progress commentary short. Return only the requested JSON, without Markdown fences."
    ),
    "tools": [mcp_tool],
}
print(json.dumps(mcp_tool, indent=2))


### 최종 응답과 파일의 계약

07장과 같은 업무 도구를 다른 연결 방식으로 호출합니다. `summary.json`에는 기본 매출 집계만 저장하고, 최종 JSON에는 `metrics`·`target_revenue`·`attainment_rate`를 반환합니다. 보고서에는 목표와 달성률도 표시합니다.

기본 `metrics` 구조와 최종 확장 응답 계약을 분리해 실제 입력에 조합합니다. 목표값은 프롬프트에 넣지 않고 MCP 도구 결과를 사용하도록 요청합니다.


In [ ]:
METRICS_CONTRACT = (
    'The metrics object must contain exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": integer, '
    '"by_product": {product_name: integer_revenue}}. '
    "Revenue is quantity multiplied by unit_price. Do not round integer amounts. "
    "Each by_product value must be integer revenue, never a nested object."
)
RESULT_CONTRACT = (
    "Return only a JSON object with keys metrics (the base object), "
    "target_revenue (integer), and attainment_rate "
    "(total_revenue / target_revenue as a unit fraction, not multiplied by 100)."
)
task = (
    "Read /workspace/sales.csv and calculate sales metrics using Python. "
    "Call get_sales_target for period 2026-01 exactly once and use the actual tool result. "
    "Write /workspace/outputs/summary.json with only the base metrics. "
    "The final metrics object must equal the parsed summary.json object. "
    "Write /workspace/outputs/report.md in Korean beginning with '# 매출 분석 보고서'; "
    "include all metrics, the target, and attainment percentage. "
    + METRICS_CONTRACT + " " + RESULT_CONTRACT
)


## 3. MCP 서버 초기화와 에이전트 실행

세션 생성 요청에 `agent`와 `environment`를 전달하면 환경 준비와 MCP 프로세스 초기화 후 분석이 진행됩니다. 노트북은 별도 MCP 클라이언트나 로컬 서버 프로세스를 만들지 않습니다.

서버 초기화 실패와 정상 연결 후 도구 호출 실패는 다른 단계입니다. 기본 실습의 예상하지 않은 외부 action은 오류로 처리하고, 실패·취소는 `check_event()`로 감지합니다. 이벤트 수·순서를 고정하거나 `idle`만 보고 성공으로 판단하지 않습니다.

각 실행·결과 단계는 오류·일반적인 인터럽트에서 정리를 시도합니다. 정상 경로에서는 **7절 정리 셀까지 실행**하고 중간에 학습을 멈출 때도 정리 셀을 실행하세요. 강제 커널 종료는 자동 정리를 보장하지 못합니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

복구 기록은 `outputs/extensions/<chapter>/<run_id>/`, 다운로드는 그 아래 `downloads/<turn_id>/`에 보관합니다. 상태 준비 셀은 새 실행 시작에 한 번만 실행하세요. 진행 중 다시 실행하면 자원 식별자를 잃을 수 있습니다.

타이머는 실제 API 실행 직전에 시작합니다. 600초 제한은 이벤트 도착 시 확인하므로 무응답 중 강제 중단하는 전체 타이머는 아닙니다. HTTP timeout은 90초입니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/extensions" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
result = turn = None
downloads = None
downloads_complete = False


### 세션 생성과 MCP 호출 관찰

MCP Call Item의 서버·도구 이름을 표시하고 스트리밍 응답을 수집합니다. 호출 Item의 도착과 성공 여부는 구분하고, 저장된 최종 상태는 다음 절에서 확인합니다. 이 셀은 새 작업을 제출하므로 중복 실행은 차단합니다.

스트림 컨텍스트를 닫은 뒤 root Turn을 조회합니다. 완료 상태가 분석값·도구 결과의 정확성까지 보장하는 것은 아닙니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent=agent,
        environment=environment,
        input=task,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.turn.item.added" and event.item.type == "mcp_call":
                print("MCP call item:", event.item.server_label, event.item.name)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted MCP exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        result = stream.get_final_result()
        sessions.setdefault(result.session_id, {"deleted": False})
    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15,
    )
    if result.turn.status != "completed" or turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nCompleted turn:", {"session_id": result.session_id, "turn_id": turn.id,
                                 "status": turn.status})


## 4. 이번 Turn의 실제 MCP 호출 이력 확인하기

`items.list()`를 SDK 페이지 반복으로 읽고 이번 root Turn의 `mcp_call`만 표시합니다. 서버 label·도구 이름·인자·Item ID·Turn ID·상태·오류·출력을 확인하세요. 출력은 이 합성 목표 조회의 결과이며 전체 HTTP 오류 body나 자격 증명을 표시하는 코드가 아닙니다.

별도 검증기는 의도한 서버와 정확한 도구명, 요청 기간, 완료 상태·오류 없음, 호출 1회를 검사합니다. 이것은 프롬프트의 요청 조건 확인이며 장애를 넘어선 exactly-once 보장이나 데이터 접근 격리의 증거가 아닙니다. 도구 호출 기록과 최종 계산값은 각각 확인합니다.

기록에 실패·오류가 있으면 정상 도구 호출로 판단하지 말고 7절 정리 후 8절의 원인 확인으로 이동하세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before inspecting MCP history.")
    items = list(client.beta.agents.sessions.items.list(
        result.session_id, order="asc", limit=100, timeout=15,
    ))
    mcp_calls = [item for item in items
                 if item.type == "mcp_call" and item.turn_id == result.turn_id]
    print("MCP calls from this turn:")
    for call in mcp_calls:
        arguments = json.loads(call.arguments) if isinstance(call.arguments, str) else call.arguments
        print({"server_label": call.server_label, "name": call.name, "arguments": arguments,
               "item_id": call.id, "turn_id": call.turn_id, "status": call.status,
               "error": call.error, "output": call.output})
    print({"get_sales_target_calls": sum(c.name == "get_sales_target" for c in mcp_calls)})


## 5. 정확한 Turn의 결과 파일 다운로드하기

`artifacts.for_result(result)`로 완료한 Session·Turn에 범위를 고정하고 정확한 경로를 다운로드합니다. SDK가 누락·중복 경로를 오류로 처리합니다. 반환된 artifact 정보와 로컬 경로·크기를 확인하세요.

두 파일을 모두 받은 뒤 완료로 표시합니다. 다운로드 재실행은 세션 정리 전에 한정합니다. 오류 시 정리를 시도하므로 같은 세션에서 재시도할 수 있다고 보장하지 않으며, 부분 파일을 성공으로 처리하거나 분석을 자동 재실행하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before downloading artifacts.")
    downloads_complete = False
    downloads = output_dir / "downloads" / result.turn_id
    downloads.mkdir(parents=True, exist_ok=True)
    scoped = client.beta.agents.sessions.artifacts.for_result(result)
    for name in ("summary.json", "report.md"):
        local_path = downloads / name
        artifact = scoped.download(f"/workspace/outputs/{name}", to=local_path, timeout=30)
        print({"artifact_id": artifact.id, "turn_id": artifact.turn_id,
               "sandbox_path": artifact.path, "local_path": str(local_path),
               "local_bytes": local_path.stat().st_size})
    downloads_complete = True


## 6. 확장 응답·기본 집계·보고서 읽기

최종 응답의 `metrics`와 기본 `summary.json`을 구분해 표시합니다. 최종 응답에는 MCP 조회를 활용한 목표와 달성률도 있습니다. JSON의 비율 `0.85`는 보고서의 백분율 `85%`와 같은 표현입니다.

아래 정상 경로는 로컬 파일을 읽고 보고서를 표시합니다. JSON 파싱·사람의 확인과 별도 검증기의 CSV/목표/보고서 정답 대조는 서로 다른 확인입니다. 다운로드한 파일은 세션 삭제 후에도 읽을 수 있습니다.


In [ ]:
from IPython.display import Markdown, display

with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not downloads_complete:
        raise RuntimeError("Complete both downloads before reading local results.")
    actual = json.loads(result.output_text)
    downloaded_summary = json.loads((downloads / "summary.json").read_text(encoding="utf-8"))
    report_text = (downloads / "report.md").read_text(encoding="utf-8")
    print("Final enriched response:")
    print(json.dumps(actual, indent=2))
    print("Downloaded base summary.json:")
    print(json.dumps(downloaded_summary, indent=2))
    display(Markdown(report_text))


## 7. 세션 정리와 로컬 결과 보관

필요한 파일을 내려받은 뒤 이번 실행의 세션을 정리합니다. 중간에 학습을 멈춰도 정리 셀을 실행하세요. 스트림 종료는 세션 삭제나 MCP 프로세스의 물리적 종료 확인과 별개입니다.

helper는 이번 실행의 metadata로 소유 세션을 찾아 필요하면 취소한 뒤 삭제·재조회 `404`로 API상 부재를 확인합니다. 출력의 `deleted`·`verified_absent`를 확인하세요. 물리적 sandbox/MCP 프로세스 정리가 동기적으로 끝났음을 별도로 검사하는 코드는 아닙니다.

원래 실행·검증 오류를 보존하면서 정리 오류도 처리합니다. 검증기는 삭제 전에 이력·파일 정답 검사를 추가합니다. 실행 사본은 `outputs/extensions-validation/`, 결과·정리 기록은 `outputs/extensions/`에 저장하며 수동 Jupyter 실행에는 receipt를 추가하지 않습니다. 로컬 결과와 다운로드는 남습니다.


In [ ]:
try:
    print("Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})


## 8. 단계별 오류 확인과 운영 경계

| 단계 | 확인할 내용 |
| --- | --- |
| 환경 준비 | 원격 패키지·공급 파일·네트워크 설정 |
| MCP 초기화 | 실행 프로그램·서버/데이터 경로·기존 절대 cwd |
| 도구 호출 | 기간 인자·서버의 검사·MCP Item 상태/오류 |
| 분석 | root Turn의 실패·취소·최종 응답 |
| 다운로드 | 정확한 Turn/경로·누락 artifact·부분 파일·정리 결과 |

`required=True`인 서버의 초기화 실패는 Turn 실패로 확인합니다. 이 노트북은 실패 이벤트를 감지하고 MCP Item의 상태·오류를 표시하지만 서버 stderr를 수집하는 별도 API는 호출하지 않습니다. 프로세스 로그 접근이 가능한 실행 환경에서는 그 로그로 초기화 원인을 추가 조사합니다. 로그가 확보되지 않았다면 stderr까지 검증했다고 기록하지 않습니다.

401/403은 인증·프로젝트 권한, 429는 API 제한을 확인합니다. 강제 종료 시 `outputs/extensions/<chapter>/<run_id>/creation.json`과 Session ID로 이번 실행만 식별하세요. 같은 생성 요청을 무조건 반복하거나 키·원본 API 오류 body를 출력하지 않습니다. 복구는 [빠른 시작](../quickstart.ipynb), 부작용 도구의 캐시·승인은 11장에서 확인합니다.


### 보조 참고: HTTP와 stdio의 연결 위치

| 연결 | 접근·실행 위치 |
| --- | --- |
| HTTP / `service` | OpenAI 서비스 네트워크에서 접근할 서버 |
| HTTP / `environment` | 선택한 실행 환경에서 접근할 서버 |
| stdio | 선택한 실행 환경에서 시작할 프로세스 |

노트북 PC의 localhost와 원격 sandbox의 localhost는 다릅니다. 본문은 stdio만 실습하며 HTTP 연결·사내 인증·네트워크 경로를 검증하지 않습니다. [공식 연결 가이드](https://developers.openai.com/api/docs/guides/agents-api/tools/mcp)


## 완료 기준과 다음 장

서버 코드·공급 파일·프로세스 시작 설정을 연결해 설명하고, 실제 MCP 호출 이력과 확장 응답·결과 파일·세션 정리 상태를 구분해 읽을 수 있어야 합니다. 서버 초기화 성공과 실제 도구 호출 성공은 별도 확인입니다.

[9장 Skills](09_skills.ipynb)에서는 분석 절차와 실행 코드를 재사용 가능한 Skill로 묶습니다.

**심화 참고:** [15장 Multi-agent](15_multi_agent.ipynb)에 MCP 목표 조회를 연결하는 확장은 별도 실습입니다. 현재 15장은 목표 JSON 파일을 사용하며 MCP와 Multi-agent 조합은 이번 기본 실행의 검증 범위가 아닙니다.


## 공식 참고 자료

### Agents API 전용 문서

- [MCP connections](https://developers.openai.com/api/docs/guides/agents-api/tools/mcp) — MCP 연결 위치·stdio/HTTP·인증·도구 접근 제어.
- [OpenAI-hosted sandboxes](https://developers.openai.com/api/docs/guides/agents-api/environments/openai-hosted) — packages·setup_commands·초기 파일·네트워크 설정.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
